# MovieLens 25M Data Acquisition

**Purpose:** Download, extract, and process MovieLens 25M dataset for DS 4320 Project 1

**Data Source:** GroupLens Research, University of Minnesota  
**URL:** https://grouplens.org/datasets/movielens/25m/  
**Date Acquired:** March 10, 2026

**Output:** 
- 7 CSV tables (1.087 GB total)
- 7 Parquet files (198 MB total)
- Relational structure with 4 core tables + 3 supplementary tables

In [1]:
"""
MovieLens 25M Data Acquisition Script
Rubric: Data Creation - Code Documentation (3 points)
"""

import pandas as pd
import os
import zipfile
import requests
from tqdm import tqdm
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('data_acquisition.log'),
        logging.StreamHandler()
    ]
)

# Create output directory
os.makedirs('data_output', exist_ok=True)

print("=" * 70)
print("MOVIELENS 25M DATASET ACQUISITION")
print("100% Real Movie Rating Data from GroupLens Research")
print("=" * 70)

MOVIELENS 25M DATASET ACQUISITION
100% Real Movie Rating Data from GroupLens Research


In [3]:
def download_file(url, destination):
    """
    Download file from URL with progress bar
    
    Args:
        url (str): URL to download from
        destination (str): Local file path to save to
        
    Returns:
        None
    """
    # try and except to download the file
    try:
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destination, 'wb') as file, tqdm(
            desc=destination,
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
        ) as progress_bar:
            for chunk in response.iter_content(chunk_size=8192):
                size = file.write(chunk)
                progress_bar.update(size)
        
        logging.info(f"Successfully downloaded {destination}")
    except Exception as e:
        logging.error(f"Error downloading file: {e}")
        raise

def download_movielens():
    """
    Download MovieLens 25M dataset from GroupLens
    Dataset: ~265 MB zip → ~3 GB unzipped
    
    Returns:
        str: Path to downloaded zip file
    """
    print("\n[1/3] Downloading MovieLens 25M dataset...")
    
    url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
    zip_path = "data_output/ml-25m.zip"
    
    # Check if already downloaded
    if os.path.exists(zip_path):
        print(f"  ✓ Zip file already exists ({os.path.getsize(zip_path) / (1024*1024):.2f} MB)")
        print("  Skipping download...")
        logging.info("Zip file already exists, skipping download")
    else:
        print("  Downloading ~265 MB zip file...")
        print("  This will take 2-5 minutes depending on internet speed...")
        download_file(url, zip_path)
        print(f"  ✓ Downloaded {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
    
    return zip_path

In [4]:
def extract_movielens(zip_path):
    """
    Extract MovieLens zip file
    
    Args:
        zip_path (str): Path to zip file
        
    Returns:
        str: Path to extracted data directory
    """
    print("\n[2/3] Extracting files...")
    print("  This will extract ~3 GB of CSV files...")
    # define the path to extract the files
    extract_path = "data_output/"
    # attempt to get info on the zip file
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            file_list = zip_ref.namelist()
            print(f"  Found {len(file_list)} files in archive")
            
            # Extract all files
            for file in file_list:
                zip_ref.extract(file, extract_path)
                print(f"    Extracted: {file}")
        
        print("  ✓ Extraction complete")
        logging.info("Successfully extracted all files")
        
    except Exception as e:
        logging.error(f"Error extracting files: {e}")
        raise
    
    return "data_output/ml-25m/"

In [5]:
def load_and_create_tables(data_dir):
    """
    Load MovieLens CSV files and create derived tables
    
    Creates 7 tables:
    1. movies - Movie metadata
    2. ratings - User ratings (PRIMARY - 25M rows)
    3. users - User statistics (DERIVED from ratings)
    4. tags - User-generated tags
    5. links - External IDs (IMDB, TMDB)
    6. genome_scores - Tag relevance scores
    7. genome_tags - Tag definitions
    
    Args:
        data_dir (str): Path to extracted data directory
        
    Returns:
        dict: Dictionary of DataFrames
    """
    print("\n[3/3] Loading and creating tables...")
    print("=" * 70)
    
    tables = {}
    
    # Load movies
    print("\nLoading movies.csv...")
    try:
        tables['movies'] = pd.read_csv(f"{data_dir}movies.csv")
        print(f"  ✓ Loaded {len(tables['movies']):,} movies")
        logging.info(f"Loaded {len(tables['movies'])} movies")
    except Exception as e:
        logging.error(f"Error loading movies: {e}")
        raise
    
    # Load ratings (LARGE - 25M rows)
    print("\nLoading ratings.csv (25M rows - this takes 30-60 seconds)...")
    try:
        tables['ratings'] = pd.read_csv(f"{data_dir}ratings.csv")
        print(f"  ✓ Loaded {len(tables['ratings']):,} ratings")
        logging.info(f"Loaded {len(tables['ratings'])} ratings")
    except Exception as e:
        logging.error(f"Error loading ratings: {e}")
        raise
    
    # Load tags
    print("\nLoading tags.csv...")
    try:
        tables['tags'] = pd.read_csv(f"{data_dir}tags.csv")
        print(f"  ✓ Loaded {len(tables['tags']):,} tags")
        logging.info(f"Loaded {len(tables['tags'])} tags")
    except Exception as e:
        logging.error(f"Error loading tags: {e}")
        raise
    
    # Load links
    print("\nLoading links.csv...")
    try:
        tables['links'] = pd.read_csv(f"{data_dir}links.csv")
        print(f"  ✓ Loaded {len(tables['links']):,} links")
        logging.info(f"Loaded {len(tables['links'])} links")
    except Exception as e:
        logging.error(f"Error loading links: {e}")
        raise
    
    # CREATE DERIVED TABLE: users (aggregated from ratings)
    print("\nCreating users table from ratings...")
    try:
        users_df = tables['ratings'][['userId']].drop_duplicates().reset_index(drop=True)
        users_df['num_ratings'] = tables['ratings'].groupby('userId').size().values
        users_df['avg_rating'] = tables['ratings'].groupby('userId')['rating'].mean().values
        users_df['first_rating_date'] = pd.to_datetime(
            tables['ratings'].groupby('userId')['timestamp'].min().values, unit='s'
        )
        users_df['last_rating_date'] = pd.to_datetime(
            tables['ratings'].groupby('userId')['timestamp'].max().values, unit='s'
        )
        tables['users'] = users_df
        
        # Save users table
        users_df.to_csv(f"{data_dir}users.csv", index=False)
        
        print(f"  ✓ Created users table with {len(users_df):,} users")
        logging.info(f"Created users table with {len(users_df)} users")
    except Exception as e:
        logging.error(f"Error creating users table: {e}")
        raise
    
    # Load genome scores (optional - adds 300MB)
    print("\nLoading genome-scores.csv...")
    try:
        tables['genome_scores'] = pd.read_csv(f"{data_dir}genome-scores.csv")
        print(f"  ✓ Loaded {len(tables['genome_scores']):,} genome scores")
        logging.info(f"Loaded {len(tables['genome_scores'])} genome scores")
    except Exception as e:
        logging.warning(f"Could not load genome scores: {e}")
        tables['genome_scores'] = pd.DataFrame()
    
    # Load genome tags (optional)
    print("\nLoading genome-tags.csv...")
    try:
        tables['genome_tags'] = pd.read_csv(f"{data_dir}genome-tags.csv")
        print(f"  ✓ Loaded {len(tables['genome_tags']):,} genome tags")
        logging.info(f"Loaded {len(tables['genome_tags'])} genome tags")
    except Exception as e:
        logging.warning(f"Could not load genome tags: {e}")
        tables['genome_tags'] = pd.DataFrame()
    
    return tables

In [7]:
def save_parquet(tables, data_dir):
    """
    Save all tables as Parquet format (Rubric requirement)
    
    Args:
        tables (dict): Dictionary of DataFrames
        data_dir (str): Output directory
        
    Returns:
        bool: Success status
    """
    print("\n" + "=" * 70)
    print("SAVING PARQUET FORMAT (RUBRIC REQUIREMENT)")
    print("=" * 70)
    
    try:
        for name, df in tables.items(): # for each table in the dictionary we save as a parquet file
            if not df.empty:
                parquet_path = f"{data_dir}{name}.parquet"
                df.to_parquet(parquet_path, index=False)
                print(f"  ✓ Saved {name}.parquet")
                logging.info(f"Saved {name}.parquet")
        
        print("✓ All parquet files saved")
        return True
    
    except ImportError: # if pyarrow is not installed
        print("PyArrow not installed")
        print("   Install with: pip install pyarrow")
        logging.warning("PyArrow not installed, parquet files not saved")
        return False
    
    except Exception as e:
        print(f"Error saving parquet: {e}")
        logging.error(f"Error saving parquet: {e}")
        return False

In [8]:
def verify_rubric_requirements(tables, data_dir):
    """
    Verify all rubric requirements are met
    
    Checks:
    - Minimum 4 tables
    - CSV and Parquet formats
    - Size > 1 GB
    - Relational structure
    
    Args:
        tables (dict): Dictionary of DataFrames
        data_dir (str): Data directory path
        
    Returns:
        tuple: (total_gb, has_parquet)
    """
    print("\n" + "=" * 70)
    print("RUBRIC REQUIREMENTS VERIFICATION")
    print("=" * 70)
    
    # Core tables
    core_tables = ['movies', 'ratings', 'users', 'tags']
    
    # Calculate file sizes
    total_size = 0
    print(f"{'File':<35} {'Rows':>12} {'Size (MB)':>12}")
    print("-" * 70)
    
    for name in core_tables:
        df = tables[name]
        csv_path = f"{data_dir}{name}.csv"
        
        if os.path.exists(csv_path):
            size_bytes = os.path.getsize(csv_path)
            size_mb = size_bytes / (1024 * 1024)
            total_size += size_bytes
            print(f"{name}.csv{'':<31} {len(df):>12,} {size_mb:>12.2f}")
    
    # Optional tables
    optional_tables = ['links', 'genome_scores', 'genome_tags']
    for name in optional_tables:
        if name in tables and not tables[name].empty:
            df = tables[name]
            csv_name = name.replace('_', '-')
            csv_path = f"{data_dir}{csv_name}.csv"
            
            if os.path.exists(csv_path):
                size_bytes = os.path.getsize(csv_path)
                size_mb = size_bytes / (1024 * 1024)
                total_size += size_bytes
                print(f"{csv_name}.csv{'':<31} {len(df):>12,} {size_mb:>12.2f}")
    
    total_mb = total_size / (1024 * 1024)
    total_gb = total_size / (1024 * 1024 * 1024)
    
    print("-" * 70)
    print(f"{'TOTAL (CSV)':<35} {'':>12} {total_mb:>12.2f} MB")
    print(f"{'':.<35} {'':>12} {total_gb:>12.3f} GB")
    
    # Check parquet
    parquet_total = 0
    has_parquet = False
    
    for name in list(tables.keys()):
        parquet_path = f"{data_dir}{name}.parquet"
        if os.path.exists(parquet_path):
            has_parquet = True
            parquet_total += os.path.getsize(parquet_path)
    
    if has_parquet:
        parquet_gb = parquet_total / (1024 * 1024 * 1024)
        print(f"\nParquet total: {parquet_gb:.3f} GB")
        savings = ((total_size - parquet_total) / total_size * 100)
        print(f"Space saved: {savings:.1f}%")
    
    # Rubric checklist
    print("\n" + "=" * 70)
    print("RUBRIC CHECKLIST")
    print("=" * 70)
    print(f"✓ Minimum 4 tables: {len(core_tables)} core")
    print(f"✓ CSV format: True")
    print(f"{'✓' if has_parquet else '✗'} Parquet format: {has_parquet}")
    print(f"✓ Size > 1 KB: True")
    print(f"✓ Size > 1 MB: True")
    print(f"{'✓' if total_gb > 1 else '✗'} Size > 1 GB: {total_gb > 1} ({total_gb:.3f} GB)")
    
    logging.info(f"Total size: {total_gb:.3f} GB, Parquet: {has_parquet}")
    
    return total_gb, has_parquet

In [9]:
def main():
    """
    Main execution pipeline
    
    Steps:
    1. Download MovieLens 25M zip file
    2. Extract CSV files
    3. Load and create derived tables
    4. Save as Parquet format
    5. Verify rubric requirements
    
    Returns:
        dict: Dictionary of all DataFrames
    """
    
    try:
        # Step 1: Download
        zip_path = download_movielens()
        
        # Step 2: Extract
        data_dir = extract_movielens(zip_path)
        
        # Step 3: Load and create tables
        tables = load_and_create_tables(data_dir)
        
        # Step 4: Save parquet
        parquet_success = save_parquet(tables, data_dir)
        
        # Step 5: Verify requirements
        total_gb, has_parquet = verify_rubric_requirements(tables, data_dir)
        
        # Summary
        print("\n" + "=" * 70)
        print("✓ DATA ACQUISITION COMPLETE")
        print("=" * 70)
        print(f"Total Size: {total_gb:.2f} GB")
        print(f"Tables Created: {len(tables)}")
        print(f"100% Real Data: ✓")
        print(f"Parquet Format: {'✓' if has_parquet else '✗'}")
        print("=" * 70)
        
        logging.info("Data acquisition completed successfully")
        
        return tables
        
    except Exception as e:
        logging.error(f"Data acquisition failed: {e}")
        raise

# Execute pipeline
tables = main()


[1/3] Downloading MovieLens 25M dataset...
  This will take 2-5 minutes depending on internet speed...


data_output/ml-25m.zip: 100%|██████████| 250M/250M [00:34<00:00, 7.57MB/s] 
2026-03-10 22:25:16,178 - INFO - Successfully downloaded data_output/ml-25m.zip


  ✓ Downloaded 249.84 MB

[2/3] Extracting files...
  This will extract ~3 GB of CSV files...
  Found 8 files in archive
    Extracted: ml-25m/
    Extracted: ml-25m/tags.csv
    Extracted: ml-25m/links.csv
    Extracted: ml-25m/README.txt
    Extracted: ml-25m/ratings.csv
    Extracted: ml-25m/genome-tags.csv


2026-03-10 22:25:18,111 - INFO - Successfully extracted all files
2026-03-10 22:25:18,229 - INFO - Loaded 62423 movies


    Extracted: ml-25m/genome-scores.csv
    Extracted: ml-25m/movies.csv
  ✓ Extraction complete

[3/3] Loading and creating tables...

Loading movies.csv...
  ✓ Loaded 62,423 movies

Loading ratings.csv (25M rows - this takes 30-60 seconds)...


...

  ✓ Loaded 15,584,448 genome scores

Loading genome-tags.csv...
  ✓ Loaded 1,128 genome tags

SAVING PARQUET FORMAT (RUBRIC REQUIREMENT)
  ✓ Saved movies.parquet


2026-03-10 22:25:27,538 - INFO - Saved ratings.parquet
2026-03-10 22:25:27,694 - INFO - Saved tags.parquet
2026-03-10 22:25:27,705 - INFO - Saved links.parquet
2026-03-10 22:25:27,738 - INFO - Saved users.parquet


  ✓ Saved ratings.parquet
  ✓ Saved tags.parquet
  ✓ Saved links.parquet
  ✓ Saved users.parquet


2026-03-10 22:25:28,275 - INFO - Saved genome_scores.parquet
2026-03-10 22:25:28,277 - INFO - Saved genome_tags.parquet
2026-03-10 22:25:28,278 - INFO - Total size: 1.087 GB, Parquet: True
2026-03-10 22:25:28,278 - INFO - Data acquisition completed successfully


  ✓ Saved genome_scores.parquet
  ✓ Saved genome_tags.parquet
✓ All parquet files saved

RUBRIC REQUIREMENTS VERIFICATION
File                                        Rows    Size (MB)
----------------------------------------------------------------------
movies.csv                                      62,423         2.90
ratings.csv                                  25,000,095       646.84
users.csv                                     162,541        10.26
tags.csv                                   1,093,360        37.01
links.csv                                      62,423         1.31
genome-scores.csv                                  15,584,448       415.00
genome-tags.csv                                       1,128         0.02
----------------------------------------------------------------------
TOTAL (CSV)                                           1113.34 MB
...................................                     1.087 GB

Parquet total: 0.198 GB
Space saved: 81.8%

RUBRIC CHECKLI